# Lekcija 8: Geometrijske transformacije

Ova lekcija objašnjava pomicanje, promjenu veličine i oblika slika: zrcaljenje, izrezivanje, rotaciju i skaliranje, a zatim i općenitiju hijerarhiju dvodimenzionalnih transformacija &mdash; **euklidskih**, **transformacija sličnosti** i **afinih** &mdash; čiji su navedeni postupci posebni slučajevi.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Testna slika bez simetrije

Crtamo slovo „F” jer nema rotacijsku ni zrcalnu simetriju &mdash; svako zrcaljenje i svaka rotacija daju vidljivo drukčiji rezultat, pa je transformacije lako razlikovati.


In [ ]:
def make_f_image(size=160):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[:] = (30, 30, 30)
    color = (255, 200, 0)
    cv2.rectangle(img, (40, 20), (65, 140), color, -1)   # vertical stroke
    cv2.rectangle(img, (40, 20), (120, 45), color, -1)   # top horizontal stroke
    cv2.rectangle(img, (40, 65), (100, 90), color, -1)   # middle horizontal stroke
    return img

img = make_f_image()
plt.imshow(img)
plt.title('Original')
plt.axis('off')
plt.show()

## Osnovne operacije

### Zrcaljenje

Funkcija `cv2.flip` zrcali sliku: oznaka `1` zrcali je vodoravno (lijevo–desno), `0` okomito (gore–dolje), a `-1` u oba smjera.


In [ ]:
flip_h = cv2.flip(img, 1)
flip_v = cv2.flip(img, 0)
flip_hv = cv2.flip(img, -1)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, im, title in zip(axes, [img, flip_h, flip_v, flip_hv],
                          ['Original', 'flip horizontal', 'flip vertical', 'flip both']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Izrezivanje

Izrezivanje je samo izdvajanje dijela polja pomoću raspona indeksa &mdash; nije potrebna funkcija biblioteke OpenCV. Izraz `image[y0:y1, x0:x1]` zadržava retke `y0..y1` i stupce `x0..x1`.


In [ ]:
cropped = img[10:100, 30:110]

plt.imshow(cropped)
plt.title(f'Cropped, shape={cropped.shape[:2]}')
plt.axis('off')
plt.show()

### Skaliranje (promjena veličine)

Funkcija `cv2.resize` mijenja dimenzije slike, po potrebi različitim faktorima za širinu i visinu. Omjer širine i visine naziva se **omjer stranica**. Jednoliko skaliranje čuva oblik i omjer stranica, dok nejednoliko skaliranje oblik rasteže ili sabija.


In [ ]:
uniform = cv2.resize(img, None, fx=0.5, fy=0.5)
stretched = cv2.resize(img, None, fx=1.5, fy=0.6)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, title in zip(axes, [img, uniform, stretched],
                          ['Original', 'Uniform scale (0.5, 0.5)', 'Non-uniform (1.5, 0.6)']):
    ax.imshow(im)
    ax.set_title(f'{title}\nshape={im.shape[:2]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Rotacija

Funkcija `cv2.getRotationMatrix2D` izrađuje matricu veličine $2\times3$ za rotaciju za proizvoljan kut oko odabranog središta, a `cv2.warpAffine` zatim tu matricu primjenjuje na sliku.


In [ ]:
h, w = img.shape[:2]
center = (w / 2, h / 2)
M = cv2.getRotationMatrix2D(center, angle=25, scale=1.0)
print('rotation matrix:\n', M)

rotated = cv2.warpAffine(img, M, (w, h))

plt.imshow(rotated)
plt.title('Rotated 25 degrees about the center')
plt.axis('off')
plt.show()

## Hijerarhija transformacija

Zrcaljenje, rotacija, skaliranje i translacija posebni su slučajevi opće matrične transformacije veličine $2\times3$

$$\begin{bmatrix}x'\\y'\end{bmatrix} = \begin{bmatrix}a & b\\c & d\end{bmatrix}\begin{bmatrix}x\\y\end{bmatrix} + \begin{bmatrix}t_x\\t_y\end{bmatrix}$$

koja se primjenjuje funkcijom `cv2.warpAffine`. Različita ograničenja dijela veličine $2\times2$ daju ugniježđenu obitelj transformacija, od najviše do najmanje ograničenih:

| Transformacija | Slobodni parametri | Oblik matrice | Čuva |
|---|---|---|---|
| **Euklidska (kruta)** | rotacija $\theta$, translacija $(t_x,t_y)$ | $\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}$ | duljine, kutove |
| **Sličnosti** | + jednoliko skaliranje $s$ | $s\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}$ | kutove, omjere duljina |
| **Afina** | bilo koja invertibilna matrica veličine $2\times2$ | $\begin{bmatrix}a & b\\c & d\end{bmatrix}$ | paralelnost, omjere duž pravca |

Kretanjem prema dnu tablice dvije se stvari mijenjaju u suprotnim smjerovima. *Skup dopuštenih transformacija* raste: svaka euklidska transformacija poseban je slučaj transformacije sličnosti, uz $s=1$, a svaka transformacija sličnosti poseban je slučaj afine transformacije, s matricom rotacije i skaliranja umjesto proizvoljne matrice. Stoga skup transformacija svakog retka obuhvaća skup iz prethodnog retka. S tom dodatnom fleksibilnošću smanjuje se broj *svojstava čije je očuvanje zajamčeno*: euklidska transformacija čuva i duljine i kutove; transformacija sličnosti odustaje od očuvanja samih duljina i čuva samo njihove omjere; afina transformacija odustaje i od očuvanja kutova. Afine transformacije najopćenitije su od ovih triju: smicanjem mogu pretvoriti kvadrat u paralelogram, što ne mogu ni euklidske transformacije ni transformacije sličnosti.


### Usporedba na jediničnom kvadratu

Najjasniji način da vidimo što svaka transformacija može, a što ne može učiniti jest promatrati što se događa s jednostavnim kvadratom.


In [ ]:
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=np.float64)

def apply_2x2(pts, A, t=(0, 0)):
    return pts @ A.T + np.array(t)

theta = np.radians(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

euclidean_sq  = apply_2x2(square, R, t=(0.3, 0.1))
similarity_sq = apply_2x2(square, 1.6 * R, t=(0.3, 0.1))
affine_sq     = apply_2x2(square, np.array([[1.6, 0.7], [0.2, 0.9]]), t=(0.3, 0.1))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, pts, title in zip(
    axes,
    [euclidean_sq, similarity_sq, affine_sq],
    ['Euclidean\n(rotate + translate)', 'Similarity\n(+ uniform scale)', 'Affine\n(shear allowed)'],
):
    ax.plot(*square.T, '--', color='gray', label='original')
    ax.plot(*pts.T, color='#e74c3c', linewidth=2, label='transformed')
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

Samo kvadrat nakon afine transformacije prestaje izgledati kao kvadrat, eventualno promijenjene veličine &mdash; njegovi kutovi više nisu pravi jer afine transformacije dopuštaju smicanje. Sve tri transformacije zadržavaju paralelnost suprotnih stranica. Općenitije projektivne transformacije, primjerice perspektiva kamere, obrađuju se u Lekciji 24.


### Određivanje afine transformacije iz parova odgovarajućih točaka

U praksi često ne znamo izravno matricu transformacije. Primjerice, ravni objekt fotografiran je *pod kutom* ili je skenirana slika ispala nakošena, a želimo ukloniti izobličenje. Potrebno je odrediti nekoliko karakterističnih točaka na izobličenoj fotografiji i mjesta na kojima bi se one *trebale* nalaziti u standardnom frontalnom prikazu. Funkcija `cv2.getAffineTransform` prima 3 takva para odgovarajućih točaka, što je najmanji broj potreban za određivanje svih 6 afinih parametara, te vraća matricu koja jedan skup preslikava u drugi. Odredimo li transformaciju iz izobličenog u standardni prikaz, njezina primjena *ispravlja* fotografiju.


In [ ]:
canvas_size = (w + 60, h + 60)

# simulate a photo taken at an angle: apply a KNOWN affine warp (rotation + shear + scale)
# -- in a real application this warp is unknown; we only ever get to see `warped` below
theta = np.radians(20)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
shear = np.array([[1, 0.25], [0, 1]])
A_true = R @ shear * 1.1
t_true = np.array([40, 20])
M_true = np.hstack([A_true, t_true.reshape(2, 1)])

warped = cv2.warpAffine(img, M_true, canvas_size)

# 3 landmarks we can identify in BOTH images (e.g. by clicking on the F's outer corners)
canonical_pts = np.float32([[40, 20], [120, 20], [40, 140]])
warped_pts = ((A_true @ canonical_pts.T).T + t_true).astype(np.float32)  # where those 3 points ended up

# solve for the warped -> canonical transform, then apply it to rectify the whole photo
M_recovered = cv2.getAffineTransform(warped_pts, canonical_pts)
rectified = cv2.warpAffine(warped, M_recovered, (w, h))

pixel_error = np.abs(rectified.astype(int) - img.astype(int)).mean()
print('mean abs pixel error, rectified vs. true original:', round(pixel_error, 2))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(img)
axes[0].set_title('Original (canonical)', fontsize=9)
axes[1].imshow(warped)
for p in warped_pts:
    axes[1].scatter(*p, c='red', s=30)
axes[1].set_title('Warped photo\n(3 identified landmarks)', fontsize=9)
axes[2].imshow(rectified)
axes[2].set_title('Rectified\n(unwarped back)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Zadaci

1. `cv2.flip` nije dio afine obitelji koju smo prethodno zapisali u matričnom obliku (matrice rotacije $R$ uvijek imaju determinantu $+1$). Koju determinantu ima matrica zrcaljenja veličine $2\times2$? Izradite matricu veličine $2\times2$ za `flip(1)` i provjerite.
2. Izradite matricu transformacije sličnosti sa $s=1$ i provjerite odgovara li čistoj euklidskoj transformaciji &mdash; odnosno da je transformacija sličnosti strogo proširenje euklidske transformacije, a ne zasebna obitelj.
3. Izmijenite afinu matricu u primjeru s jediničnim kvadratom tako da suprotne stranice *nisu* paralelne. Koji uvjet za to mora prestati vrijediti? (Pomoć: to je upravo granica koja razdvaja afine od projektivnih transformacija.)
